In [ ]:
import pandas as pd
%load_ext autoreload
%autoreload 2

from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import napari
import colorcet as cc
from tqdm import tqdm

import dnt

spots_directory = Path(r"C:\Tracking\BlastodermAnalysis\data\spots")
embryo_overview = pd.read_excel(spots_directory / "overview.xlsx", sheet_name="Sheet1")
save_path = Path(r"C:\Tracking\BlastodermAnalysis\figures\figure_4")


embryo_overview = embryo_overview[embryo_overview["good"]]
included = embryo_overview["Embryo"].astype(str).tolist()
condition_map = {
    str(embryo): condition for embryo, condition in zip(embryo_overview["Embryo"], embryo_overview["condition"])
}
print(condition_map)

dnt.set_plot_style()
spots_dfs, stems = dnt.load_spots_data(spots_directory, included)

print(stems)

df = spots_dfs[0]
cycles = [10, 11, 12, 13, 14]

print(df.columns)

greens = ["#143601","#1a4301","#245501","#538d22","#73a942","#aad576"][::-1]
blues = ["#012a4a","#01497c","#2a6f97","#468faf","#89c2d9"][::-1]
reds = ["#641220","#85182a","#a71e34","#bd1f36", "#da1e37"][::-1]

all_mmfs = {}

for k in range(len(spots_dfs)):
    stem = stems[k]
    df = spots_dfs[k]

    min_mvmt_frames, times = dnt.find_stationary_timepoints(df)
    all_mmfs[stem] = min_mvmt_frames

In [ ]:
def hex2rgb(color):
    return int(color[1:3], 16) / 255, int(color[3:5], 16) / 255, int(color[5:7], 16) / 255

region_colors = np.array(["#0a9396", "#ee9b00", "#ae2012"])
base_color = r"#e9d8a6"
region_aps = [(0.05, 0.25), (0.4, 0.6), (0.75, 0.95)]
region_names = ["Anterior", "Middle", "Posterior"]
displacement_limits = (-6, 10)

condition_pal = {
    "wt": blues[2],
    "bcd": reds[2],
    "trk": greens[2],
}

ap_vals = np.linspace(0.02, 0.98, 25)

### Figure 4a:
Density comparison between trk and wt at nc 10, 12, 14

In [ ]:
import dnt
import pandas as pd
from collections import defaultdict

all_mmfs = {}

for k in range(len(spots_dfs)):
    stem = stems[k]
    df = spots_dfs[k]

    min_mvmt_frames, times = dnt.find_stationary_timepoints(df)
    all_mmfs[stem] = min_mvmt_frames

all_surface_areas = dnt.calculate_density.calculate_all_surface_areas(spots_dfs, stems, all_mmfs, ap_vals)
cycle_relative_densities = dnt.calculate_density.calculate_relative_densities(spots_dfs, stems, all_mmfs, all_surface_areas, condition_map, cycles)

In [ ]:
def get_plotting_df(relative_densities_df):
    # remove most anterior and posterior positions (pole cells)
    plotting_df = relative_densities_df.query("positions > 0.01 and positions < 0.96").copy()

    # apply rolling mean for each density profile
    plotting_df["densities_local_smooth"] = (plotting_df.groupby(["cycle", "source"])["densities"]
                                                 .transform(lambda x: x.rolling(5, center=True, min_periods=1).mean()))

    return plotting_df

def plot_compare_conditions_at_cycle(relative_densities_df, cycle, conditions, ax, legend=False):
    plotting_df = get_plotting_df(relative_densities_df)

    filtered_df = plotting_df.query("cycle == @cycle and condition in @conditions")

    for k in filtered_df["source"].unique():
        source_subset = filtered_df[filtered_df["source"] == k].copy()
        sns.lineplot(source_subset, x="positions", y="densities_local_smooth", hue="condition", palette=condition_pal, errorbar=None, alpha=0.3, legend=False, ax=ax)

    sns.lineplot(filtered_df, x="positions", y="densities_local_smooth", hue="condition", palette=condition_pal, lw=3, legend=legend, ax=ax, errorbar=None)


fig, axes = plt.subplots(1, 5, figsize=(12, 1.5), sharey=True)
for cycle, ax in zip([10, 11, 12, 13, 14], axes):
    conditions = ["wt", "trk"]
    plot_compare_conditions_at_cycle(cycle_relative_densities, cycle, conditions, ax, legend=False)
    ax.set_xlabel("AP position")
    ax.set_xticks([0, 0.5, 1.0])
    ax.set_title(f"Cycle {cycle}")
    ax.spines[["top", "right"]].set_visible(False)

axes[0].set_ylim(0.55, 1.3)
axes[0].set_ylabel("Relative Density")
plt.savefig(save_path / f"nc_5cycle_density_{"".join(conditions)}_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(9, 1.5), sharey=True)
for cycle, ax in zip([10,  12, 14], axes):
    # fig, ax = plt.subplots(1, 1, figsize=(4, 2.5))
    conditions = ["wt", "trk"]
    plot_compare_conditions_at_cycle(cycle_relative_densities, cycle, conditions, ax, legend=False)
    # plt.legend(loc="lower right")
    ax.set_xlabel("AP position")
    ax.set_xticks([0, 0.5, 1.0])
    # ax.set_ylabel("Relative density")
    # ax.set_ylim(0.5, 1.2)
    ax.set_title(f"Cycle {cycle}")
    ax.spines[["top", "right"]].set_visible(False)

axes[0].set_ylim(0.55, 1.3)
axes[0].set_ylabel("Relative Density")
plt.savefig(save_path / f"nc_3cycle_density_{"".join(conditions)}_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

### 4b Bands figure wt

In [ ]:
import pandas as pd
from tqdm import tqdm
from collections import defaultdict


df = spots_dfs[0]

mesh_path = save_path / "all_meshes"
mesh_path.mkdir(exist_ok=True)

colors = [hex2rgb(c) for c in region_colors]

track_id_colors = {}

for i, region in enumerate(region_aps):
    track_ids = df.query("frame < 30 and AP.between(@region[0], @region[1])")["track_id"].unique()

    for track_id in track_ids:
        track_id_colors[track_id] = colors[i]

for i, frame in tqdm(enumerate(df["frame"].unique())):

    frame_df = df.query("frame == @frame")

    points = frame_df[["z", "y", "x"]].values
    mesh = dnt.mesh_from_points(points)

    mesh.write_obj(mesh_path / f"frame_{frame}.obj")
    track_ids = frame_df["track_id"].values

    colors = [track_id_colors.get(tid, hex2rgb(base_color)) for tid in track_ids]

    valid = pd.Series(np.arange(len(points))).isin(np.unique(mesh.faces))
    blender_save = pd.DataFrame(np.array(colors)[valid], columns=["R", "G", "B"])
    blender_save["track_id"] = track_ids[valid]

    blender_save.to_csv(mesh_path / f"frame_{frame}_colors.csv", index=False)

### 4b Bands figure trk

In [ ]:
import pandas as pd
from tqdm import tqdm
from collections import defaultdict


df = spots_dfs[7]

mesh_path = save_path / "all_meshes_trk"
mesh_path.mkdir(exist_ok=True)

colors = [hex2rgb(c) for c in region_colors]

track_id_colors = {}

for i, region in enumerate(region_aps):
    track_ids = df.query("frame < 30 and AP.between(@region[0], @region[1])")["track_id"].unique()

    for track_id in track_ids:
        track_id_colors[track_id] = colors[i]

for i, frame in tqdm(enumerate(df["frame"].unique())):

    frame_df = df.query("frame == @frame")

    points = frame_df[["z", "y", "x"]].values
    mesh = dnt.mesh_from_points(points)

    mesh.write_obj(mesh_path / f"frame_{frame}.obj")
    track_ids = frame_df["track_id"].values

    colors = [track_id_colors.get(tid, hex2rgb(base_color)) for tid in track_ids]

    valid = pd.Series(np.arange(len(points))).isin(np.unique(mesh.faces))
    blender_save = pd.DataFrame(np.array(colors)[valid], columns=["R", "G", "B"])
    blender_save["track_id"] = track_ids[valid]

    blender_save.to_csv(mesh_path / f"frame_{frame}_colors.csv", index=False)

### Figure 4c: AP displacement over time for the 3 regions in wt and trk

In [ ]:
k = 0

for k in [7, 8, 9, 10]:
    df = spots_dfs[k]

    t = df.groupby(["track_id", "frame"])[["x", "y", "z", "time_since_nc11", "AP", "theta", "cycle"]].mean().reset_index()

    t["AP_from_start"] = t["AP"] - t["track_id"].map(t.groupby("track_id")["AP"].first())
    t["AP_from_start_percent"] = t["AP_from_start"] * 100
    t["time_since_nc11"] = np.round(t["time_since_nc11"], 3)

    fig, axes = plt.subplots(figsize=(4.5, 2.2))

    for i, ap_group in enumerate(region_aps):

        region_track_ids = t[t["AP"].between(*ap_group)]["track_id"].unique()
        early_track_ids = t.groupby("track_id")["time_since_nc11"].min() < 0
        early_track_ids = t["track_id"].unique()[early_track_ids]

        all_good = np.intersect1d(region_track_ids, early_track_ids)
        t_new = t[t["track_id"].isin(all_good)].copy()

        sns.lineplot(t_new, x="time_since_nc11", y="AP_from_start_percent", color=region_colors[i], errorbar=None, alpha=1, label=region_names[i], linewidth=4, legend=False)

        for cycle in cycles[1:]:
            cycle_df = t[t["cycle"] == cycle]
            cycle_times = cycle_df.groupby("track_id")["time_since_nc11"].min()
            division_time = cycle_times.median()
            plt.axvline(division_time, color="k", linestyle="--", linewidth=2, alpha=0.2)

        axes.spines["top"].set_visible(False)
        axes.spines["right"].set_visible(False)

    axes.invert_yaxis()
    plt.title(f"Average displacement of region over time")
    plt.ylabel("Ap displacement (% Embryo length)")
    plt.xlabel("Time since nc11 (minutes)")
    plt.ylim(displacement_limits)

    plt.savefig(save_path / f"{stems[k]}_nuclear_movement_over_time.png", dpi=300, bbox_inches="tight")
    plt.show()